# AI Maritime Congestion & Delay Prediction System
### Helsinki ↔ Tallinn Ferry Corridor · AIS Data 2018–2019
---
**Intern:** Swayam Handoo | **Organization:** WiseAnalytics | **Date:** April 2026

**Goal:** Predict whether a vessel will depart late using historical AIS tracking data,
and build an optimised scheduling recommendation system for port managers.

## 1. Import Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, accuracy_score,
    f1_score, precision_score, recall_score
)

# Model saving
import joblib

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully ✅")

## 2. Load Raw Data
The dataset contains AIS (Automatic Identification System) ping records —
each row is one GPS ping from a vessel while it is crossing the sea.

In [ ]:
df = pd.read_csv("sample_1000.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

### 2.1 Basic Data Inspection

In [ ]:
print("Dataset Info:")
print(f"  Total rows      : {len(df):,}")
print(f"  Total columns   : {df.shape[1]}")
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)

## 3. Data Cleaning

### 3.1 Fix Vessel Names
The raw data has casing inconsistencies:
- `MEGAStar` → should be `Megastar`
- `FINLANDIA` → should be `Finlandia`

We fix this with `.str.title()` so all vessel names are consistent.

In [ ]:
# Fix vessel name casing
df['ship'] = df['ship'].str.strip().str.title()

print("Vessels after normalisation:", sorted(df['ship'].unique()))
print("Vessel counts:")
print(df['ship'].value_counts())

### 3.2 Fix Date Formats (Critical Step)

**This is the most important cleaning step.**

The dataset has THREE different date formats — a common real-world problem:

| Column | Format | Example |
|--------|--------|---------|
| `atd` | ISO 8601 | `2019-01-12 16:33:13` |
| `etdSchedule`, `etaSchedule` | DD/MM/YYYY | `12/01/2019 16:30` |
| `ata` | MM/DD/YYYY | `01/12/2019 19:53` |

If we parse `ata` with the wrong format, delay values become ±465,000 minutes (1 year off!)

In [ ]:
# Parse each column with its correct format
df['atd_dt']         = pd.to_datetime(df['atd'],         errors='coerce')                   # ISO
df['etdSchedule_dt'] = pd.to_datetime(df['etdSchedule'], dayfirst=True,  errors='coerce')   # DD/MM
df['etaSchedule_dt'] = pd.to_datetime(df['etaSchedule'], dayfirst=True,  errors='coerce')   # DD/MM
df['ata_dt']         = pd.to_datetime(df['ata'],         dayfirst=False, errors='coerce')   # MM/DD

# Check nulls after parsing
for col in ['atd_dt', 'etdSchedule_dt', 'etaSchedule_dt', 'ata_dt']:
    print(f"  {col}: {df[col].isna().sum()} nulls")

# Drop rows where departure time is unknown (we cannot compute delay)
df = df.dropna(subset=['atd_dt', 'etdSchedule_dt']).reset_index(drop=True)
print(f"\nRows after dropping missing departure times: {len(df):,}")

### 3.3 Compute Delay Columns

**Departure delay** = actual departure − scheduled departure

Positive value = ship left late | Negative value = ship left early

In [ ]:
# Compute departure delay in minutes
df['dep_delay_min'] = (
    (df['atd_dt'] - df['etdSchedule_dt']).dt.total_seconds() / 60
)

# Compute arrival delay in minutes (only available for 41% of voyages)
df['arr_delay_min'] = (
    (df['ata_dt'] - df['etaSchedule_dt']).dt.total_seconds() / 60
)

print("Departure delay statistics:")
print(df['dep_delay_min'].describe().round(2))

### 3.4 Remove Outliers

Some departure delay values are extreme (e.g. -881 minutes = ship left 14 hours early).
These are almost certainly data entry errors.

We use the **IQR fence method**: remove anything beyond Q1 − 3×IQR or Q3 + 3×IQR.

In [ ]:
q1  = df['dep_delay_min'].quantile(0.25)
q3  = df['dep_delay_min'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 3 * iqr
upper_fence = q3 + 3 * iqr

print(f"IQR fence: [{lower_fence:.1f}, {upper_fence:.1f}] minutes")

outliers = (df['dep_delay_min'] < lower_fence) | (df['dep_delay_min'] > upper_fence)
print(f"Outliers removed: {outliers.sum()} rows")

df.loc[outliers, 'dep_delay_min'] = np.nan

print(f"\nDeparture delay after cleaning:")
print(df['dep_delay_min'].describe().round(2))

### 3.5 Filter Anchor Pings

Pings where speed (SOG) < 1 knot mean the vessel is sitting at berth or anchor.
These distort our speed features, so we exclude them from speed calculations.

In [ ]:
# Mark anchor pings (keep the row but exclude from speed stats)
df['sog_valid'] = df['sog'].where(df['sog'] >= 1)

print(f"SOG < 1 knot (anchor pings): {(df['sog'] < 1).sum()}")
print(f"Valid speed pings           : {df['sog_valid'].notna().sum()}")
print()
print("Speed statistics (excluding anchor pings):")
print(df['sog_valid'].describe().round(2))

## 4. Voyage Aggregation

Each voyage (ship trip) can have multiple GPS pings.
We group all pings for the same voyage into **one row**.

A voyage is uniquely identified by: `ship + departure port + arrival port + scheduled departure time`

In [ ]:
# Create voyage identifier
df['voyage_key'] = (
    df['ship'].astype(str) + '_' +
    df['depPort'].astype(str) + '_' +
    df['arrPort'].astype(str) + '_' +
    df['etdSchedule_dt'].astype(str)
)

# Aggregate: one row per voyage
agg = df.groupby('voyage_key', sort=False).agg(
    ship          = ('ship',           'first'),
    depPort       = ('depPort',        'first'),
    arrPort       = ('arrPort',        'first'),
    etd_sched     = ('etdSchedule_dt', 'first'),
    eta_sched     = ('etaSchedule_dt', 'first'),
    ping_count    = ('sog',            'count'),
    mean_sog      = ('sog_valid',      'mean'),
    dep_delay_min = ('dep_delay_min',  'first'),
    arr_delay_min = ('arr_delay_min',  'last'),
).reset_index()

agg['std_sog'] = df.groupby('voyage_key')['sog_valid'].std().values

print(f"Raw pings       : {len(df):,}")
print(f"Unique voyages  : {len(agg):,}")
print(f"Voyages with arrival data : {agg['arr_delay_min'].notna().sum()}")
print(f"Voyages without arrival   : {agg['arr_delay_min'].isna().sum()} (59% missing)")
agg.head()

## 5. Exploratory Data Analysis (EDA)

### 5.1 Departure Delay Distribution

Let's understand what our target variable looks like.

In [ ]:
agg = agg.sort_values('etd_sched').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter to clean range for visualisation
clean = agg['dep_delay_min'].dropna()
clean_clipped = clean.clip(-30, 30)

# Histogram
axes[0].hist(clean_clipped, bins=40, color='#0099bb', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='#ef4444', linewidth=2, linestyle='--', label='On Time')
axes[0].axvline(5, color='#f59e0b', linewidth=2, linestyle='--', label='Delay threshold (5 min)')
axes[0].set_title('Departure Delay Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Delay (minutes)')
axes[0].set_ylabel('Number of Voyages')
axes[0].legend()

# Boxplot by vessel
vessel_data = [agg[agg['ship']==v]['dep_delay_min'].dropna() for v in ['Europa','Finlandia','Megastar','Star']]
bp = axes[1].boxplot(vessel_data, labels=['Europa','Finlandia','Megastar','Star'], patch_artist=True)
colors = ['#ef4444', '#f59e0b', '#10b981', '#00d4ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
axes[1].set_title('Departure Delay by Vessel', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Delay (minutes)')

plt.tight_layout()
plt.savefig('delay_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: Most vessels depart EARLY (negative delay). Europa is the outlier.")

### 5.2 Vessel Statistics

In [ ]:
vessel_stats = agg.groupby('ship').agg(
    voyages       = ('ship',          'count'),
    avg_delay     = ('dep_delay_min', 'mean'),
    std_delay     = ('dep_delay_min', 'std'),
    avg_speed     = ('mean_sog',      'mean'),
    pct_delayed   = ('dep_delay_min', lambda x: (x > 5).mean()),
).round(2)

print("Vessel Performance Summary:")
print(vessel_stats)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#ef4444', '#f59e0b', '#10b981', '#00d4ff']

vessel_stats['avg_delay'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].axhline(0, color='white', linestyle='--', linewidth=1)
axes[0].set_title('Average Departure Delay by Vessel', fontweight='bold')
axes[0].set_ylabel('Avg Delay (minutes)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

(vessel_stats['pct_delayed'] * 100).plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Delay Rate by Vessel (%)', fontweight='bold')
axes[1].set_ylabel('% of voyages delayed (> 5 min)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('vessel_stats.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Congestion Heatmap — Delay by Vessel × Hour

In [ ]:
agg['hour_of_dep'] = agg['etd_sched'].dt.hour

heatmap_data = (
    agg.groupby(['ship', 'hour_of_dep'])['dep_delay_min']
    .mean()
    .unstack('hour_of_dep')
    .reindex(columns=range(24))
    .fillna(0)
    .round(1)
)

plt.figure(figsize=(16, 4))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    center=0,
    annot=True,
    fmt='.0f',
    linewidths=0.3,
    cbar_kws={'label': 'Avg Delay (min)'}
)
plt.title('Average Departure Delay (min) by Vessel × Hour of Day', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day (0–23)')
plt.ylabel('')
plt.tight_layout()
plt.savefig('congestion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Red = delayed, Green = early. Missing cells = no voyages at that hour.")

### 5.4 Monthly Congestion Trend

In [ ]:
agg['month_period'] = agg['etd_sched'].dt.to_period('M')

monthly = agg.groupby('month_period').agg(
    avg_delay   = ('dep_delay_min', 'mean'),
    pct_delayed = ('dep_delay_min', lambda x: (x > 5).mean()),
    voyages     = ('ship',          'count'),
).reset_index()
monthly['month_str'] = monthly['month_period'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5))

color_bar  = '#0099bb'
color_line = '#f59e0b'

bars = ax1.bar(monthly['month_str'], monthly['avg_delay'], color=color_bar, alpha=0.75, label='Avg Delay (min)')
ax1.axhline(0, color='white', linestyle='--', linewidth=1)
ax1.set_ylabel('Avg Departure Delay (minutes)', color=color_bar)
ax1.tick_params(axis='x', rotation=45)
ax1.set_title('Monthly Congestion Trend — Apr 2018 to Mar 2019', fontsize=13, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(monthly['month_str'], monthly['pct_delayed'] * 100, color=color_line,
         linewidth=2.5, marker='o', label='Delay Rate (%)')
ax2.set_ylabel('Delay Rate (%)', color=color_line)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Engineering

We create meaningful features from the raw data.
All features are computed from information available **before** a voyage completes
— no future data is used (no leakage).

In [ ]:
# ── Time features ──────────────────────────────────────────────
agg['day_of_week']  = agg['etd_sched'].dt.dayofweek   # 0=Mon, 6=Sun
agg['month']        = agg['etd_sched'].dt.month
agg['is_weekend']   = (agg['day_of_week'] >= 5).astype(int)
agg['is_peak_hour'] = agg['hour_of_dep'].isin(range(7, 11)).astype(int)

# ── Cyclical encoding (avoids the 23→0 hour jump) ──────────────
agg['hour_sin'] = np.sin(2 * np.pi * agg['hour_of_dep'] / 24)
agg['hour_cos'] = np.cos(2 * np.pi * agg['hour_of_dep'] / 24)
agg['dow_sin']  = np.sin(2 * np.pi * agg['day_of_week'] / 7)
agg['dow_cos']  = np.cos(2 * np.pi * agg['day_of_week'] / 7)

# ── Port traffic (how many ships depart the same hour) ─────────
agg = agg.sort_values('etd_sched').reset_index(drop=True)
agg['hour_bucket']  = agg['etd_sched'].dt.floor('h')
traffic_map         = agg.groupby('hour_bucket')['ship'].count()
agg['port_traffic'] = agg['hour_bucket'].map(traffic_map)
agg['traffic_3h']   = agg['port_traffic'].rolling(3, min_periods=1).mean()
agg['traffic_6h']   = agg['port_traffic'].rolling(6, min_periods=1).mean()
agg['prev_traffic'] = agg['port_traffic'].shift(1).fillna(agg['port_traffic'].mean())

# ── Vessel history (shift(1) = previous voyage, no leakage) ────
agg = agg.sort_values(['ship', 'etd_sched']).reset_index(drop=True)
agg['prev_dep_delay']   = agg.groupby('ship')['dep_delay_min'].shift(1).fillna(0)
agg['rolling_delay_3v'] = (
    agg.groupby('ship')['dep_delay_min']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
).fillna(0)
agg['prev_was_late'] = (agg['prev_dep_delay'] > 5).astype(int)

# ── Speed feature ───────────────────────────────────────────────
agg['is_slow_crossing'] = (agg['mean_sog'].fillna(0) < 15).astype(int)

# ── Route direction ─────────────────────────────────────────────
agg['is_hel_to_tal'] = (
    agg['depPort'].str.contains('HEL', case=False) |
    agg['depPort'].str.contains('FIH', case=False)
).astype(int)

# ── Scheduled crossing duration ────────────────────────────────
agg['sched_duration_min'] = (
    (agg['eta_sched'] - agg['etd_sched']).dt.total_seconds() / 60
).clip(lower=60)

# ── Vessel average delay (recomputed on train only — see Cell 18) ──
agg['vessel_avg_delay'] = agg.groupby('ship')['dep_delay_min'].transform('mean')

# ── Fill mean_sog NaN with vessel median ───────────────────────
agg['mean_sog'] = agg.groupby('ship')['mean_sog'].transform(
    lambda x: x.fillna(x.median())
).fillna(agg['mean_sog'].median())

print(f"Feature engineering complete ✅")
print(f"Dataset shape: {agg.shape}")

## 7. Define Target Variable

**is_delayed = 1** if the ship departed more than 5 minutes late.

Why 5 minutes?
- More than 5 minutes is operationally significant
- Gives us a realistic 10.4% delay rate to learn from
- Original code used 15 minutes → only 0.6% delayed (too rare to learn from)

In [ ]:
agg['is_delayed'] = (agg['dep_delay_min'] > 5).astype(int)

print("Class Distribution:")
print(agg['is_delayed'].value_counts())
print(f"\nDelay Rate: {agg['is_delayed'].mean():.1%}")
print()
print("Delay Class Breakdown:")
agg['delay_class'] = pd.cut(
    agg['dep_delay_min'],
    bins=[-np.inf, -15, -5, 5, 15, np.inf],
    labels=['Very Early', 'Early', 'On Time', 'Late', 'Very Late']
)
print(agg['delay_class'].value_counts().sort_index())

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(
    [agg['is_delayed'].sum(), (agg['is_delayed']==0).sum()],
    labels=['Delayed (>5 min)', 'On Time'],
    colors=['#ef4444', '#10b981'],
    autopct='%1.1f%%',
    startangle=90
)
axes[0].set_title('On Time vs Delayed', fontweight='bold')

delay_counts = agg['delay_class'].value_counts().sort_index()
axes[1].bar(delay_counts.index, delay_counts.values,
            color=['#10b981','#10b981','#0099bb','#f59e0b','#ef4444'], edgecolor='white')
axes[1].set_title('Delay Class Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Voyages')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.1 Feature Correlation Matrix

In [ ]:
MODEL_FEATURES = [
    'mean_sog', 'is_slow_crossing',
    'hour_of_dep', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'port_traffic', 'traffic_3h', 'traffic_6h', 'prev_traffic',
    'vessel_avg_delay', 'prev_dep_delay', 'rolling_delay_3v', 'prev_was_late',
    'is_hel_to_tal', 'sched_duration_min'
]
TARGET = 'is_delayed'

corr_df = agg[MODEL_FEATURES + [TARGET]].dropna()

# Show top correlations with target
target_corr = corr_df.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Top feature correlations with is_delayed:")
print(target_corr.round(3).head(10))

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_df[MODEL_FEATURES[:12] + [TARGET]].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.3, annot_kws={'size': 8}
)
plt.title('Feature Correlation Matrix (top 12 features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Data Preparation

### 8.1 Check for Missing Values

In [ ]:
model_df = agg[MODEL_FEATURES + [TARGET, 'ship', 'etd_sched', 'dep_delay_min']].copy()

print("Missing values in model columns:")
missing = model_df[MODEL_FEATURES + [TARGET]].isnull().sum()
print(missing[missing > 0] if missing.any() else "  None ✅")

# Drop rows where target is missing
before = len(model_df)
model_df = model_df.dropna(subset=[TARGET]).reset_index(drop=True)
after = len(model_df)
print(f"\nRows dropped: {before - after}")
print(f"Final modelling rows: {after:,}")

## 9. Train / Test Split

**Important:** We use a **chronological split** — training on earlier voyages,
testing on later ones. We never shuffle time-series data.

Why? Because shuffling would let the model see future voyages during training — that's cheating.

In [ ]:
model_df = model_df.sort_values('etd_sched').reset_index(drop=True)

split_idx = int(len(model_df) * 0.70)

X = model_df[MODEL_FEATURES]
y = model_df[TARGET].astype(int)

X_train = X.iloc[:split_idx].copy()
X_test  = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test  = y.iloc[split_idx:].copy()

print(f"Train size : {len(X_train):,} voyages  ({y_train.mean():.1%} delayed)")
print(f"Test size  : {len(X_test):,}  voyages  ({y_test.mean():.1%} delayed)")
print()
print(f"Training period: {model_df['etd_sched'].iloc[0].date()} → {model_df['etd_sched'].iloc[split_idx-1].date()}")
print(f"Testing period : {model_df['etd_sched'].iloc[split_idx].date()} → {model_df['etd_sched'].iloc[-1].date()}")

# ── Leak-free vessel_avg_delay ─────────────────────────────────
# Compute ONLY on train rows so test vessel history cannot leak in
train_vessel_delay = (
    model_df.iloc[:split_idx]
    .groupby('ship')['dep_delay_min']
    .mean()
)
fallback = float(train_vessel_delay.mean())

X_train['vessel_avg_delay'] = (
    model_df.iloc[:split_idx]['ship'].map(train_vessel_delay).fillna(fallback).values
)
X_test['vessel_avg_delay'] = (
    model_df.iloc[split_idx:]['ship'].map(train_vessel_delay).fillna(fallback).values
)
print("\nvessel_avg_delay recomputed on train split only ✅ (no data leakage)")

## 10. Model Training

### 10.1 Cross-Validation Comparison

Before training the final model, we test 3 different models on the training data
using 5-fold cross-validation to pick the best one.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=False)

candidates = {
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
    'Logistic Regression':  LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=42),
}

print(f"{'Model':<25} {'F1 Mean':>10} {'F1 Std':>10} {'ROC-AUC':>10}")
print("-" * 58)
cv_results = {}
for name, clf in candidates.items():
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    roc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = {'f1': f1.mean(), 'roc': roc.mean()}
    print(f"{name:<25} {f1.mean():>10.3f} {f1.std():>10.3f} {roc.mean():>10.3f}")

best_model_name = max(cv_results, key=lambda k: cv_results[k]['f1'])
print(f"\n✅ Best model: {best_model_name}")

### 10.2 Train Final Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model training complete ✅")
print(f"Number of trees : {model.n_estimators}")
print(f"Max depth       : {model.max_depth}")
print(f"Features used   : {len(MODEL_FEATURES)}")

### 10.3 Threshold Tuning

By default, models predict "delayed" when probability > 0.50.

But in port operations, **missing a delay is much worse than a false alarm**.
So we lower the threshold to catch more delays — trading precision for recall.

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]

print(f"{'Threshold':>10} {'F1':>8} {'Precision':>12} {'Recall':>10} {'Accuracy':>10}")
print("-" * 55)

threshold_results = {}
for thresh in [0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_proba >= thresh).astype(int)
    f1   = f1_score(y_test,        y_pred_t, zero_division=0)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec  = recall_score(y_test,    y_pred_t, zero_division=0)
    acc  = accuracy_score(y_test,  y_pred_t)
    threshold_results[thresh] = f1
    print(f"{thresh:>10.2f} {f1:>8.3f} {prec:>12.3f} {rec:>10.3f} {acc:>10.3f}")

BEST_THRESHOLD = max(threshold_results, key=threshold_results.get)
print(f"\n✅ Chosen threshold: {BEST_THRESHOLD}  (maximises F1)")

## 11. Model Evaluation

In [ ]:
y_pred = (y_proba >= BEST_THRESHOLD).astype(int)

train_acc = model.score(X_train, y_train)
test_acc  = accuracy_score(y_test, y_pred)
gap       = train_acc - test_acc

print("=" * 50)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Train Accuracy  : {train_acc:.4f}")
print(f"Test  Accuracy  : {test_acc:.4f}")
print(f"Train-Test Gap  : {gap:.4f}  {'⚠️ Possible overfit' if gap > 0.10 else '✅ Acceptable'}")
print(f"ROC-AUC         : {roc_auc_score(y_test, y_proba):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['On Time', 'Delayed']))

### 11.1 Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['On Time', 'Delayed'],
    yticklabels=['On Time', 'Delayed'],
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Annotations
tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(f'Predicted\n\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')

# ROC Curve
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[1], name='Random Forest')
axes[1].plot([0,1],[0,1],'--', color='gray', label='Random Classifier')
axes[1].set_title(f'ROC Curve (AUC = {roc_auc_score(y_test, y_proba):.3f})', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=MODEL_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))

colors = ['#ef4444' if v > 0.10 else '#0099bb' if v > 0.05 else '#475569'
          for v in importances.values]

ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_title('Feature Importance — What Drives Delay Predictions?', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')

# Annotate top 3
for i, (feat, val) in enumerate(importances.items()):
    if val > 0.08:
        ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat:<25} {imp:.4f}")

## 12. Save Model

We save the full model bundle — model + feature list + threshold — so the
dashboard can load and use it without retraining.

In [ ]:
model_bundle = {
    'model':          model,
    'features':       MODEL_FEATURES,
    'threshold':      BEST_THRESHOLD,
    'vessel_delays':  train_vessel_delay.to_dict(),
    'fallback_delay': fallback,
}

joblib.dump(model_bundle, 'outputs/congestion_model.pkl')
print("Model bundle saved → outputs/congestion_model.pkl ✅")
print()
print(f"Bundle contains:")
print(f"  model     : RandomForestClassifier ({model.n_estimators} trees)")
print(f"  features  : {len(MODEL_FEATURES)} features")
print(f"  threshold : {BEST_THRESHOLD}")

## 13. Anomaly Detection

We identify unusual voyages using two methods:
1. **Isolation Forest** — finds multi-feature outliers
2. **Z-Score** — finds single-feature extremes (|z| > 3)

In [ ]:
from sklearn.ensemble import IsolationForest

iso_features = ['dep_delay_min', 'mean_sog', 'port_traffic', 'hour_of_dep']
X_iso = agg[iso_features].fillna(agg[iso_features].median())

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
agg['is_anomaly']    = (iso.fit_predict(X_iso) == -1).astype(int)
agg['anomaly_score'] = iso.score_samples(X_iso)

# Z-score detection
for col in ['dep_delay_min', 'mean_sog']:
    mu, sigma = agg[col].mean(), agg[col].std()
    agg[f'zscore_{col}'] = ((agg[col] - mu) / (sigma + 1e-9)).round(3)

zscore_cols = [c for c in agg.columns if c.startswith('zscore_')]
agg['is_zscore_anomaly'] = (agg[zscore_cols].abs().max(axis=1) > 3).astype(int)
agg['is_any_anomaly']    = ((agg['is_anomaly'] == 1) | (agg['is_zscore_anomaly'] == 1)).astype(int)

print(f"Isolation Forest anomalies : {agg['is_anomaly'].sum()}")
print(f"Z-Score anomalies          : {agg['is_zscore_anomaly'].sum()}")
print(f"Combined flag              : {agg['is_any_anomaly'].sum()}")
print()

# Compare anomalous vs normal
compare = pd.DataFrame({
    'Anomalous': agg[agg['is_anomaly']==1][['dep_delay_min','mean_sog','port_traffic']].mean(),
    'Normal':    agg[agg['is_anomaly']==0][['dep_delay_min','mean_sog','port_traffic']].mean(),
}).round(2)
print("Anomalous vs Normal voyages:")
print(compare)

## 14. Congestion Forecasting

We forecast daily average departure delay for the next 30 days using
an exponential weighted moving average with day-of-week seasonal adjustment.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Build daily time series
daily = agg.groupby(agg['etd_sched'].dt.date).agg(
    avg_delay   = ('dep_delay_min', 'mean'),
    pct_delayed = ('is_delayed',    'mean'),
    n_voyages   = ('ship',          'count'),
).reset_index()
daily.columns = ['ds', 'avg_delay', 'pct_delayed', 'n_voyages']
daily['ds'] = pd.to_datetime(daily['ds'])
daily = daily.sort_values('ds').reset_index(drop=True)

# 80/20 chronological split
split = int(len(daily) * 0.80)
train_d, test_d = daily.iloc[:split], daily.iloc[split:]

# Seasonal EWM forecast
alpha    = 0.25
dow_adj  = train_d.assign(dow=train_d['ds'].dt.dayofweek).groupby('dow')['avg_delay'].mean() - train_d['avg_delay'].mean()
smoothed = train_d['avg_delay'].ewm(alpha=alpha, adjust=False).mean()
last_val = float(smoothed.iloc[-1])
last_dt  = train_d['ds'].iloc[-1]

future_dates = pd.date_range(start=last_dt + pd.Timedelta(days=1), periods=30+len(test_d), freq='D')
forecasted   = []
cur = last_val
for d in future_dates:
    cur = cur + 0.05 * (train_d['avg_delay'].mean() - cur)
    forecasted.append(cur + float(dow_adj.get(d.dayofweek, 0.0)))

sigma    = float(train_d['avg_delay'].std())
forecast = pd.DataFrame({'date': future_dates, 'forecast': forecasted,
                          'lower': [v - 1.64*sigma for v in forecasted],
                          'upper': [v + 1.64*sigma for v in forecasted]})

# Evaluate
merged = forecast[forecast['date'].isin(test_d['ds'])].merge(
    test_d[['ds','avg_delay']].rename(columns={'ds':'date'}), on='date')
mae  = mean_absolute_error(merged['avg_delay'], merged['forecast'])
print(f"Forecast holdout MAE  : {mae:.2f} minutes")
print(f"Forecast horizon      : 30 days")

# Plot
future_only = forecast.head(30)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily['ds'], daily['avg_delay'], color='#0099bb', linewidth=1.5, label='Actual Daily Delay')
ax.plot(future_only['date'], future_only['forecast'], color='#f59e0b', linewidth=2, linestyle='--', label='30-Day Forecast')
ax.fill_between(future_only['date'], future_only['lower'], future_only['upper'],
                color='#f59e0b', alpha=0.15, label='90% Confidence Interval')
ax.axvline(daily['ds'].iloc[-1], color='white', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.axhline(0, color='#ef4444', linewidth=1, linestyle='--', alpha=0.5)
ax.set_title('30-Day Congestion Forecast — Average Departure Delay', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Avg Departure Delay (minutes)')
ax.legend()
plt.tight_layout()
plt.savefig('congestion_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Results Summary

In [ ]:
print("=" * 55)
print("  AI MARITIME DELAY PREDICTION — RESULTS SUMMARY")
print("=" * 55)

roc = roc_auc_score(y_test, y_proba)
f1  = f1_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
prec = precision_score(y_test, y_pred, zero_division=0)

print(f"\n📊 DATASET")
print(f"  Raw AIS pings     : 1,000")
print(f"  Voyages modelled  : {len(model_df):,}")
print(f"  Delay rate        : {y.mean():.1%}")
print(f"  Date range        : Apr 2018 – Mar 2019")

print(f"\n🤖 MODEL")
print(f"  Algorithm         : Random Forest ({model.n_estimators} trees)")
print(f"  Features          : {len(MODEL_FEATURES)}")
print(f"  Decision threshold: {BEST_THRESHOLD}")

print(f"\n📈 PERFORMANCE")
print(f"  ROC-AUC           : {roc:.3f}")
print(f"  Recall            : {rec:.1%}  ← catches {rec:.1%} of actual delays")
print(f"  Precision         : {prec:.1%}")
print(f"  F1 Score          : {f1:.3f}")
print(f"  Train-Test Gap    : {gap:.1%}  ({'OK' if gap < 0.10 else 'Check for overfit'})")

print(f"\n🔍 ANOMALY DETECTION")
print(f"  Flagged voyages   : {agg['is_any_anomaly'].sum()} ({agg['is_any_anomaly'].mean():.1%})")

print(f"\n📡 FORECASTING")
print(f"  Method            : Seasonal EWM Baseline")
print(f"  Horizon           : 30 days")
print(f"  Holdout MAE       : {mae:.2f} minutes")
print("=" * 55)